# Day 3: Pandas Refresher
Topics: groupby, merge, pivot, filtering — tools used constantly in this project

In [2]:
import pandas as pd
import numpy as np

# Tiny mock data — mirrors what our real employees.csv will look like
employees = pd.DataFrame({
    "employee_id": ["E001", "E002", "E003", "E004", "E005", "E006"],
    "name":        ["Aarav", "Priya", "Rohan", "Sneha", "Dev", "Meera"],
    "role":        ["Backend Dev", "Data Engineer", "Backend Dev",
                    "Frontend Dev", "Data Engineer", "Frontend Dev"],
    "experience_years": [5, 3, 7, 2, 6, 4],
    "availability_pct": [80, 100, 60, 100, 50, 80],
    "cost_band":   ["B", "A", "C", "A", "C", "B"],
})

projects = pd.DataFrame({
    "project_id":   ["P001", "P002", "P003"],
    "project_name": ["FinTech App", "ML Pipeline", "E-Commerce Site"],
    "required_role": ["Backend Dev", "Data Engineer", "Frontend Dev"],
    "min_experience": [4, 2, 3],
    "deadline_days": [30, 60, 45],
})

print("Employees:"); print(employees)
print("\nProjects:");  print(projects)

Employees:
  employee_id   name           role  experience_years  availability_pct  \
0        E001  Aarav    Backend Dev                 5                80   
1        E002  Priya  Data Engineer                 3               100   
2        E003  Rohan    Backend Dev                 7                60   
3        E004  Sneha   Frontend Dev                 2               100   
4        E005    Dev  Data Engineer                 6                50   
5        E006  Meera   Frontend Dev                 4                80   

  cost_band  
0         B  
1         A  
2         C  
3         A  
4         C  
5         B  

Projects:
  project_id     project_name  required_role  min_experience  deadline_days
0       P001      FinTech App    Backend Dev               4             30
1       P002      ML Pipeline  Data Engineer               2             60
2       P003  E-Commerce Site   Frontend Dev               3             45


## 1. Filtering

In [3]:
# Filter employees available >= 80% 
available = employees[employees["availability_pct"] >= 80]
print("Available >= 80%:")
print(available[["name", "role", "availability_pct"]])

# Filter by role AND experience
senior_backend = employees[
    (employees["role"] == "Backend Dev") &
    (employees["experience_years"] >= 5)
]
print("\nSenior Backend Devs (5+ yrs):")
print(senior_backend[["name", "experience_years", "availability_pct"]])

Available >= 80%:
    name           role  availability_pct
0  Aarav    Backend Dev                80
1  Priya  Data Engineer               100
3  Sneha   Frontend Dev               100
5  Meera   Frontend Dev                80

Senior Backend Devs (5+ yrs):
    name  experience_years  availability_pct
0  Aarav                 5                80
2  Rohan                 7                60


## 2. GroupBy

In [ ]:
# Average experience and availability by role
by_role = employees.groupby("role").agg(
    headcount        = ("employee_id",     "count"),
    avg_experience   = ("experience_years","mean"),
    avg_availability = ("availability_pct","mean"),
    fully_available  = ("availability_pct", lambda x: (x == 100).sum())
).reset_index()

print("Role Summary:")
print(by_role.round(1))

# Multi-key groupby: role + cost_band
by_role_cost = employees.groupby(["role", "cost_band"]).agg(
    count = ("employee_id", "count"),
    avg_exp = ("experience_years", "mean")
).reset_index()

print("\nRole × Cost Band:")
print(by_role_cost)

Role Summary:
            role  headcount  avg_experience  avg_availability  fully_available
0    Backend Dev          2             6.0              70.0                0
1  Data Engineer          2             4.5              75.0                1
2   Frontend Dev          2             3.0              90.0                1

Role × Cost Band:
            role cost_band  count  avg_exp
0    Backend Dev         B      1      5.0
1    Backend Dev         C      1      7.0
2  Data Engineer         A      1      3.0
3  Data Engineer         C      1      6.0
4   Frontend Dev         A      1      2.0
5   Frontend Dev         B      1      4.0


## 3. Merge (JOIN)

In [ ]:
# Find eligible employees for each project
# Step 1: cross join employees × projects
employees["_key"] = 1
projects["_key"]  = 1
cross = employees.merge(projects, on="_key").drop("_key", axis=1)

# Step 2: keep rows where role matches AND experience qualifies
eligible = cross[
    (cross["role"] == cross["required_role"]) &
    (cross["experience_years"] >= cross["min_experience"]) &
    (cross["availability_pct"] >= 60)
].copy()

print("Eligible Employee-Project Pairs:")
print(eligible[["name","role","experience_years",
                "availability_pct","project_name","deadline_days"]])

## 4. Pivot Table

In [ ]:
# Availability by role and cost band — like a staffing heat map
pivot = employees.pivot_table(
    values  = "availability_pct",
    index   = "role",
    columns = "cost_band",
    aggfunc = "mean",
    fill_value = 0
)

print("Avg Availability % by Role × Cost Band:")
print(pivot)

# Skill coverage pivot: how many people per role are >= 80% available
coverage = employees[employees["availability_pct"] >= 80].pivot_table(
    values  = "employee_id",
    index   = "role",
    aggfunc = "count",
    fill_value = 0
).rename(columns={"employee_id": "available_headcount"})

print("\nAvailable Headcount per Role (>=80%):")
print(coverage)

## 5. Apply + Lambda (custom column creation)

In [ ]:
# Create a simple readiness score: experience * (availability/100)
employees["readiness_score"] = employees.apply(
    lambda row: round(row["experience_years"] * (row["availability_pct"] / 100), 2),
    axis=1
)

print("Employees with Readiness Score (exp × availability):")
print(employees[["name","role","experience_years",
                 "availability_pct","readiness_score"]]
      .sort_values("readiness_score", ascending=False))

##  Day 3 Summary
- `[]` filtering with conditions → candidate shortlisting
- `groupby().agg()` → role/skill summaries
- `merge()` cross join + filter → eligibility matching (preview of matcher.py)
- `pivot_table()` → availability heatmaps
- `apply()` → custom scoring columns

These exact patterns appear in matcher.py, optimizer.py, and the dashboard.